In [ ]:
import os
import sys

# add src and config folder in the parent directory of this notebook to the path
notebook_path = os.path.abspath('')
sys.path.append(os.path.abspath(os.path.join(notebook_path, '..', 'src')))
sys.path.append(os.path.abspath(os.path.join(notebook_path, '..', 'config')))

In [ ]:
from track_manager import TrackManager
from track_manager_config import TrackManagerConfig as config

from camera_movement_estimator import CameraMovementEstimator
from team_assigner import TeamAssigner
from player_ball_assigner import PlayerBallAssigner
from view_transformer import ViewTransformer

team_assigner = TeamAssigner()

# Initialize with default value or check if the parameter exists first
player_ball_assigner = PlayerBallAssigner()

camera_movement_estimator = CameraMovementEstimator()

view_transformer  =  ViewTransformer()

# Initialize the TrackManager with the specified model path from the config
track_manager = TrackManager(
    model_path=config.model_path,
    team_assigner=team_assigner,
    player_ball_assigner=player_ball_assigner,
    camera_movement_estimator=camera_movement_estimator ,
    view_transformer=view_transformer
)

In [ ]:
import supervision as sv
from utils.video_utils import read_video, save_video

video_frames = read_video(config.input_video_path)
tracker = sv.ByteTrack()

track_manager.initialize(tracker, video_frames)

track_manager.add_positions()
track_manager.add_adjust_positions()
track_manager.add_transformed_position()
track_manager.interpolate_ball_positions()
track_manager.add_speed_and_distance()

track_manager.assign_team_colors()
track_manager.assign_ball_to_players()

track_manager.draw_annotations()
track_manager.draw_team_ball_control()
track_manager.draw_speed_and_distance()
track_manager.draw_camera_movement()

save_video(track_manager.frames, config.output_video_path)

print(f"Processed video saved to: {config.output_video_path}")